## Initalizing

In [1]:
import numpy as np
from sklearn.datasets import make_blobs

In [34]:
class logistic_regression_model:
    def __init__(self, learning_rate=1e-5, max_iterations=1000, print_cost=True):
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.print_cost = print_cost

        self.weights = None
        self.current_cost = None
        self.last_cost = None
        self.tolerance = 1e-5
        
    def hypothesis(self, X, W):
        W = W.reshape((1, self.n_features))  # shape (2,1)
        return 1 / (1 + np.exp(-np.dot(X, W.T)))
    
    def update_cost(self, Y, Yhat):
        epsilon = 1e-8
        Yhat = np.clip(Yhat, epsilon, 1 - epsilon)
        self.current_cost = -np.sum(Y * np.log(Yhat) + (1 - Y) * np.log(1 - Yhat))
        return self.current_cost
    
    def cost_by_weights(self, weights):
        Yhat = self.hypothesis(self.X, weights)
        return self.update_cost(self.Y, Yhat)
    
    def gradient(self, weights, delta=1e-5):
        # Ensure x is a numpy array for numerical operations
        weights = np.array(weights, dtype=float)
        # Initialize the gradient vector with zeros
        gradient = np.zeros_like(weights)
        
        # Iterate over each dimension (variable) of the input point x
        for i in range(len(weights)):
            # Create a small change vector for the current dimension
            weights_plus_delta = weights.copy()
            weights_minus_delta = weights.copy()
            
            weights_plus_delta[i] += delta
            weights_minus_delta[i] -= delta
            
            # Approximate the partial derivative using the central difference formula
            # (f(x + delta_i) - f(x - delta_i)) / (2 * delta)
            partial_derivative = (self.cost_by_weights(weights_plus_delta) - self.cost_by_weights(weights_minus_delta)) / (2 * delta)
            gradient[i] = partial_derivative
        
        return gradient

    def hessian(self, weights, epsilon=1e-5):
        x = np.array(weights, dtype=float)
        
        n = len(x)
        hessian = np.zeros((n, n))

        for i in range(n):
            x_plus_i = x.copy()
            x_minus_i = x.copy()
            x_plus_i[i] += epsilon
            x_minus_i[i] -= epsilon

            # Calculate the gradient at x + epsilon in the i-th direction
            grad_plus = self.gradient(x_plus_i)
            # Calculate the gradient at x - epsilon in the i-th direction
            grad_minus = self.gradient(x_minus_i)

            # Approximate the second derivative (the i-th row of the Hessian)
            # by taking the difference of the gradients and dividing by 2*epsilon
            hessian[i, :] = (grad_plus - grad_minus) / (2 * epsilon)

        return hessian
    
    def gradient_descent(self, X, Y, Yhat):
        self.weights = self.weights + self.learning_rate * np.dot((Y - Yhat).T, X)
        return self.weights
    
    def fit(self, X, Y):
        X = np.hstack([np.ones((X.shape[0], 1)), X])

        self.n_samples, self.n_features = X.shape
        
        self.weights = np.ones((1, self.n_features))
        self.last_cost = 0
        
        for it in range(self.max_iterations):
            Yhat = self.hypothesis(X, self.weights)
            cost = self.update_cost(Y, Yhat)
            
            if self.print_cost:
                print(f"Cost at iteration {it} is {cost}")
                
            if abs(cost - self.last_cost) <= 1e-3:
                break
            self.last_cost = cost
                
            self.weights = self.gradient_descent(X, Y, Yhat)

        return self.weights
    
    def newtons_method(self, X, Y):
        X = np.hstack([np.ones((X.shape[0], 1)), X])
        self.X = X
        self.Y = Y

        self.n_samples, self.n_features = X.shape
        self.weights = np.zeros((self.n_features,))

        for i in range(self.max_iterations):
            grad = self.gradient(self.weights)
            hess = self.hessian(self.weights)

            self.weights -= np.linalg.inv(hess).dot(grad)
            
            Yhat = self.hypothesis(X, self.weights)
            cost = self.update_cost(self.Y, Yhat)
            if self.print_cost:
                print(f"Cost at iteration {i} is {cost}")

            if np.linalg.norm(grad) < self.tolerance:
                break

        return self.weights

    def predict(self, X):
        X = np.hstack([np.ones((X.shape[0], 1)), X])
        print(self.weights)

        return (self.hypothesis(X, self.weights) >= 0.5).astype(int)


## Generate data

In [10]:
np.random.seed(1)
X, y = make_blobs(n_samples=1000, centers=2)
y = y[:, np.newaxis]
print(X.shape, y.shape)

(1000, 2) (1000, 1)


## Training

In [37]:
clf = logistic_regression_model(max_iterations=10000, learning_rate=1e-5, print_cost=True)
# clf.fit(X, y)
clf.newtons_method(X, y)

y_predict = clf.predict(X)
y_predict = y_predict.reshape(-1, 1)

print(f"Accuracy: {np.sum(y==y_predict)/X.shape[0]}")

Cost at iteration 0 is 139.43971765497773
Cost at iteration 1 is 49.49007481887271
Cost at iteration 2 is 18.644516217119854
Cost at iteration 3 is 7.180009605278227
Cost at iteration 4 is 2.7968884993373244
Cost at iteration 5 is 1.097875043650265
Cost at iteration 6 is 0.4333568619234488
Cost at iteration 7 is 0.1716511387658457
Cost at iteration 8 is 0.06804751588319875
Cost at iteration 9 is 0.026908436990667486
Cost at iteration 10 is 0.010583393119335479
Cost at iteration 11 is 0.004131121138722941
Cost at iteration 12 is 0.001602089457061185
Cost at iteration 13 is 0.0006178278143384267
Cost at iteration 14 is 0.00023607133368847623
Cost at iteration 15 is 9.154418879366693e-05
Cost at iteration 16 is 0.0006710581167157161
Cost at iteration 17 is 0.0002592399159115491
Cost at iteration 18 is 0.00010444356497238284
Cost at iteration 19 is 4.5655602390599556e-05
Cost at iteration 20 is 2.399022374182567e-05
Cost at iteration 21 is 1.539849393298013e-05
Cost at iteration 22 is 1.20